# decision-binder: optional Colab LoRA wrapper
Select **Runtime → Change runtime type → NVIDIA GPU** (T4 or better). This notebook calls the repository's Python implementation; it contains no duplicate trainer.

**Run all executes only a two-step smoke test.** Full training stays disabled unless you explicitly change `RUN_FULL_TRAINING` below. Test data is never passed to training. The repository URL must be filled in after publishing this local project; no Git remote is currently configured. Use an immutable repository commit for reproduction.


In [ ]:
REPO_URL = ""  # REQUIRED: URL of your published decision-binder repository
REPO_REF = "main"  # Prefer an immutable commit SHA
USE_GOOGLE_DRIVE = True
RUN_FULL_TRAINING = False  # Change only when you intend to approve a full run


In [ ]:
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError("This project requires Python >=3.12; select a compatible Colab runtime.")
if not REPO_URL or REPO_URL.startswith("-"):
    raise ValueError("Set REPO_URL to your published repository URL first.")
repo = Path("/content/decision-binder")
if not repo.exists():
    subprocess.run(["git", "clone", "--", REPO_URL, str(repo)], check=True)
elif not (repo / ".git").exists():
    raise RuntimeError("Existing directory is not a Git checkout; choose a fresh runtime.")
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print("Repository commit:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--extra", "training", "--python", sys.executable], cwd=repo, check=True)
python = str(repo / ".venv/bin/python")


In [ ]:
# Verify the actual training environment and GPU, not the notebook kernel's packages.
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([python, "-c", "import torch; print('Torch:', torch.__version__); assert torch.cuda.is_available(), 'CUDA unavailable'; print('GPU:', torch.cuda.get_device_name()); print('Memory GiB:', torch.cuda.get_device_properties(0).total_memory/1024**3); print('Native bf16:', torch.cuda.is_bf16_supported(including_emulation=False))"], cwd=repo, check=True)


## Prepared datasets
If prepared files are unavailable, run the existing deterministic preparation pipeline. This may generate all three splits for provenance, but the training script opens **only train and validation**. No test evaluation runs here. No raw datasets, model weights, or adapters need to be committed to Git.


In [ ]:
required = [
    "data/sft/primary/train.parquet", "data/sft/primary/validation.parquet",
    "results/sft/primary/report.json",
]
if not all((repo / name).exists() for name in required):
    for command in [
        ["scripts/01_prepare_data.py"],
        ["scripts/02_create_splits.py"],
        ["scripts/02_build_sft_dataset.py", "--download-tokenizer"],
    ]:
        subprocess.run([python, *command], cwd=repo, check=True)
else:
    print("Prepared data found. Training will validate its checksums before use.")


In [ ]:
from datetime import datetime, timezone
import json

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    storage = Path("/content/drive/MyDrive/decision-binder/runs")
else:
    storage = Path("/content/decision-binder-runs")
    print("WARNING: /content is temporary. Download the output before the runtime disconnects.")
run_dir = storage / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "notebook_context.json").write_text(json.dumps({"repository_url": REPO_URL, "commit": commit, "full_training_approved": RUN_FULL_TRAINING}, indent=2))
print("Persistent output:", run_dir)


## GPU smoke test (8 train examples, 4 validation examples, 2 optimizer steps)
The same CLI verifies completion masks, checks trainable LoRA parameters, logs loss, checks finite updates, saves an adapter, and reloads it into a fresh base. It records measured peak GPU memory. A passing smoke run is a prerequisite, not a claim that every GPU/runtime combination has already been tested. Do not tune anything using the held-out test set.


In [ ]:
smoke_dir = run_dir / "smoke"
subprocess.run([python, "scripts/04_train_lora.py", "--mode", "smoke", "--download-model", "--output-dir", str(smoke_dir)], cwd=repo, check=True)
smoke = json.loads((smoke_dir / "metadata.json").read_text())
assert smoke["status"] == "passed" and smoke["adapter_reload_verified"]
print("Training loss:", smoke["train_metrics"]["train_loss"])
print("Validation loss:", smoke["validation_metrics"]["eval_loss"])
print("Peak GPU memory (GiB):", smoke["peak_gpu_allocated_gib"])
print("Adapter:", smoke_dir / "adapter")


## Full experiment — disabled by default
Only enable `RUN_FULL_TRAINING` after reviewing the smoke result and explicitly deciding to train. This starts a **fresh base + new LoRA adapter**, not the smoke adapter. Checkpoints, adapter files, progress logs, and metadata are saved directly to the chosen output location. The full run is one epoch using the repository YAML configuration.


In [ ]:
if RUN_FULL_TRAINING:
    assert smoke["status"] == "passed", "Pass the smoke test first."
    full_dir = run_dir / "full"
    subprocess.run([python, "scripts/04_train_lora.py", "--mode", "full", "--approve-full-run", "--output-dir", str(full_dir)], cwd=repo, check=True)
    full = json.loads((full_dir / "metadata.json").read_text())
    print("Full-run training metrics:", full["train_metrics"])
    print("Full-run validation metrics:", full["validation_metrics"])
    print("Saved adapter:", full_dir / "adapter")
else:
    print("Stopped after smoke test. Full training was NOT launched.")
